# Hidden Embezzlement Investigation (Phase 4)

This notebook is a forensic investigation lab for students.

## Investigation Goals
1. Load the Medici ledger dataset.
2. Explore transaction and branch-level activity.
3. Focus on Florence expense behavior.
4. Identify suspicious vendors with Benford and concentration tests.
5. Estimate likely fraud amount and date range.

In [1]:
from pathlib import Path
import math

import pandas as pd
from IPython.display import display

In [2]:
DATA_PATH = Path('data/medici_transactions.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Missing dataset: {DATA_PATH.resolve()}')

df = pd.read_csv(DATA_PATH)
df.columns = [c.strip() for c in df.columns]

df['date'] = pd.to_datetime(df['date'], errors='coerce')
for col in ['debit_amount', 'credit_amount', 'credit_amount_2']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

df['counterparty'] = df.get('counterparty', '').fillna('').astype(str).str.strip()
df['branch'] = df.get('branch', '').fillna('').astype(str).str.strip()
df['type'] = df.get('type', '').fillna('').astype(str).str.strip()
df['debit_account'] = df.get('debit_account', '').fillna('').astype(str).str.strip()
df['description'] = df.get('description', '').fillna('').astype(str).str.strip()

print(f'Loaded {len(df):,} rows from {DATA_PATH}')
print(f"Date range in file: {df['date'].min().date()} to {df['date'].max().date()}")

Loaded 85,000 rows from data/medici_transactions.csv
Date range in file: 1390-01-01 to 1440-12-31


## 1) Explore Transaction Counts

In [3]:
print('Overall transaction count:', f'{len(df):,}')
print('Unique branches:', df['branch'].nunique())
print('Unique counterparties:', df['counterparty'].nunique())

branch_counts = df['branch'].value_counts().rename_axis('branch').reset_index(name='transactions')
type_counts = df['type'].value_counts().rename_axis('type').reset_index(name='transactions')

display(branch_counts.head(15))
display(type_counts.head(15))

Overall transaction count: 85,000
Unique branches: 11
Unique counterparties: 79


,branch,transactions
0,Rome,13017
1,Florence,10941
2,Venice,8609
3,Milan,7978
4,London,7956
5,Bruges,7869
6,Avignon,7755
7,Geneva,7752
8,Pisa,6697
9,Naples,6425


,type,transactions
0,branch_activity,14325
1,recurring_operating_expense,13146
2,loan_repayment,11761
3,loan_issuance,11658
4,trade_transaction,11584
5,vendor_payment,6525
6,deposit,6162
7,war_financing,2629
8,operating_expense,1926
9,bill_of_exchange,1602


## 2) Analyze Florence Branch Expenses

We focus on expense-like transactions in Florence because hidden embezzlement is often buried in operational spend.

In [4]:
expense_types = {'operating_expense', 'recurring_operating_expense', 'vendor_payment'}
expense_accounts = {
    'Wages', 'Rent', 'Maintenance', 'Courier Services', 'Supplies',
    'Security', 'Travel', 'Entertainment', 'Marketing',
    'Depreciation', 'Miscellaneous Expense'
}

florence = df[df['branch'] == 'Florence'].copy()
florence_expenses = florence[
    florence['type'].isin(expense_types) | florence['debit_account'].isin(expense_accounts)
].copy()

florence_expenses = florence_expenses[florence_expenses['debit_amount'] > 0].copy()
florence_expenses['month'] = florence_expenses['date'].dt.to_period('M').astype(str)
florence_expenses['vendor'] = florence_expenses['counterparty'].replace('', '(blank counterparty)')

print('Florence rows:', f'{len(florence):,}')
print('Florence expense rows:', f'{len(florence_expenses):,}')
print('Florence expense total:', f
)

Florence rows: 10,941
Florence expense rows: 2,206


NameError: name 'f' is not defined

## 3) Group Expenses by Vendor

In [5]:
vendor_summary = (
    florence_expenses
    .groupby('vendor', as_index=False)
    .agg(
        transactions=('id', 'count'),
        total_amount=('debit_amount', 'sum'),
        first_date=('date', 'min'),
        last_date=('date', 'max'),
    )
)

total_expense_amount = vendor_summary['total_amount'].sum()
vendor_summary['share'] = vendor_summary['total_amount'] / total_expense_amount
vendor_summary = vendor_summary.sort_values('total_amount', ascending=False).reset_index(drop=True)

display(vendor_summary.head(20))

,vendor,transactions,total_amount,first_date,last_date,share
0,Mercato Maintenance Works,248,2199577.23,1390-06-17,1440-10-14,0.136009
1,Arno Lamp Oil Merchants,261,2120284.44,1390-01-29,1440-12-05,0.131106
2,Guildhall Security Company,257,1975651.98,1390-04-24,1440-06-05,0.122163
3,Santa Maria Scribes,241,1963349.16,1390-01-24,1440-11-14,0.121402
4,San Lorenzo Couriers,243,1956222.61,1390-04-01,1440-12-28,0.120961
5,Florentine Paperworks,247,1831857.29,1390-01-23,1440-12-19,0.113271
6,Ponte Vecchio Rent Office,249,1753266.14,1390-01-12,1440-12-31,0.108412
7,Signoria Utilities Office,208,1353255.88,1390-04-20,1440-10-16,0.083677
8,Florence Operations,252,1018852.87,1390-02-18,1440-08-05,0.063000


## 4) Apply Benford's Law

Benford expected probability for first digit $d$ is $P(d)=\log_{10}(1+1/d)$.

In [6]:
def first_significant_digit(value: float):
    text = f'{abs(value):.2f}'
    for ch in text:
        if ch.isdigit() and ch != '0':
            return int(ch)
    return None

def benford_expected():
    return {d: math.log10(1 + 1 / d) for d in range(1, 10)}

def benford_metrics(amounts):
    digits = [first_significant_digit(a) for a in amounts if a > 0]
    digits = [d for d in digits if d is not None]
    n = len(digits)
    expected = benford_expected()
    if n == 0:
        return {'n': 0, 'mad': 0.0, 'chi2': 0.0}

    observed_counts = {d: 0 for d in range(1, 10)}
    for d in digits:
        observed_counts[d] += 1

    mad = sum(abs((observed_counts[d] / n) - expected[d]) for d in range(1, 10)) / 9
    chi2 = sum(((observed_counts[d] - n * expected[d]) ** 2) / (n * expected[d]) for d in range(1, 10))

    return {'n': n, 'mad': mad, 'chi2': chi2, 'observed_counts': observed_counts}

overall_benford = benford_metrics(florence_expenses['debit_amount'].tolist())
print('Florence expense Benford metrics:')
print(overall_benford)

Florence expense Benford metrics:
{'n': 2206, 'mad': 0.008647583245061862, 'chi2': 21.18685225449277, 'observed_counts': {1: 706, 2: 396, 3: 234, 4: 228, 5: 162, 6: 143, 7: 125, 8: 135, 9: 77}}


In [7]:
vendor_benford_rows = []
for vendor, group in florence_expenses.groupby('vendor'):
    metrics = benford_metrics(group['debit_amount'].tolist())
    if metrics['n'] < 25:
        continue
    vendor_benford_rows.append({
        'vendor': vendor,
        'benford_n': metrics['n'],
        'benford_mad': metrics['mad'],
        'benford_chi2': metrics['chi2'],
    })

vendor_benford = pd.DataFrame(vendor_benford_rows).sort_values('benford_mad', ascending=False)
display(vendor_benford.head(20))

,vendor,benford_n,benford_mad,benford_chi2
7,Santa Maria Scribes,241,0.022311,15.412372
6,San Lorenzo Couriers,243,0.018429,8.209836
8,Signoria Utilities Office,208,0.017824,6.489388
1,Florence Operations,252,0.017486,11.820767
4,Mercato Maintenance Works,248,0.016163,9.040102
0,Arno Lamp Oil Merchants,261,0.013975,6.968494
2,Florentine Paperworks,247,0.013851,8.875419
5,Ponte Vecchio Rent Office,249,0.013158,5.031168
3,Guildhall Security Company,257,0.012914,6.127721


## 5) Apply Vendor Concentration Analysis

Rule-of-thumb alert levels:
- Medium: vendor share > 5%
- High: vendor share >= 20%

In [8]:
acct_vendor = (
    florence_expenses
    .groupby(['debit_account', 'vendor'], as_index=False)
    .agg(
        amount=('debit_amount', 'sum'),
        tx_count=('id', 'count')
    )
)

acct_totals = (
    florence_expenses
    .groupby('debit_account', as_index=False)
    .agg(account_total=('debit_amount', 'sum'))
)

acct_vendor = acct_vendor.merge(acct_totals, on='debit_account', how='left')
acct_vendor['share'] = acct_vendor['amount'] / acct_vendor['account_total']
acct_vendor['severity'] = 'NONE'
acct_vendor.loc[acct_vendor['share'] > 0.05, 'severity'] = 'MEDIUM'
acct_vendor.loc[acct_vendor['share'] >= 0.20, 'severity'] = 'HIGH'

concentration_flags = acct_vendor[acct_vendor['severity'] != 'NONE'].sort_values('share', ascending=False)
display(concentration_flags.head(30))

,debit_account,vendor,amount,tx_count,account_total,share,severity
54,Wages,Florence Operations,603433.08,53,1544958.26,0.390582,HIGH
31,Rent,Ponte Vecchio Rent Office,236267.31,40,1090272.84,0.216705,HIGH
12,Courier Services,Mercato Maintenance Works,155732.47,29,894613.13,0.174078,MEDIUM
24,Maintenance,Santa Maria Scribes,187228.11,34,1112278.40,0.168328,MEDIUM
35,Security,Arno Lamp Oil Merchants,176564.28,35,1079933.84,0.163495,MEDIUM
11,Courier Services,Guildhall Security Company,144583.10,32,894613.13,0.161615,MEDIUM
10,Courier Services,Florentine Paperworks,140921.22,30,894613.13,0.157522,MEDIUM
50,Supplies,San Lorenzo Couriers,128170.63,30,824403.33,0.155471,MEDIUM
17,Maintenance,Arno Lamp Oil Merchants,169050.26,29,1112278.40,0.151986,MEDIUM
3,Accounts Payable,Mercato Maintenance Works,1432953.14,85,9625857.80,0.148865,MEDIUM


## 6) Identify Suspicious Supplier

We combine three signals:
1. Vendor spend share in Florence expenses
2. Benford deviation (MAD)
3. Concentration flags within debit accounts

In [9]:
risk = vendor_summary[['vendor', 'transactions', 'total_amount', 'share']].copy()

if not vendor_benford.empty:
    risk = risk.merge(vendor_benford[['vendor', 'benford_mad', 'benford_chi2']], on='vendor', how='left')
else:
    risk['benford_mad'] = 0.0
    risk['benford_chi2'] = 0.0

flag_count = (
    concentration_flags
    .groupby('vendor', as_index=False)
    .agg(concentration_flags=('severity', 'count'))
)
risk = risk.merge(flag_count, on='vendor', how='left')
risk['concentration_flags'] = risk['concentration_flags'].fillna(0)

risk['share_rank'] = risk['share'].rank(pct=True)
risk['mad_rank'] = risk['benford_mad'].fillna(0).rank(pct=True)
risk['flag_rank'] = risk['concentration_flags'].rank(pct=True)

risk['risk_score'] = (
    0.50 * risk['share_rank'] +
    0.30 * risk['mad_rank'] +
    0.20 * risk['flag_rank']
)

risk = risk.sort_values('risk_score', ascending=False).reset_index(drop=True)
display(risk.head(15))

suspicious_supplier = risk.loc[0, 'vendor']
print('Suspicious supplier candidate:', suspicious_supplier)

,vendor,transactions,total_amount,share,benford_mad,benford_chi2,concentration_flags,share_rank,mad_rank,flag_rank,risk_score
0,Mercato Maintenance Works,248,2199577.23,0.136009,0.016163,9.040102,7,1.000000,0.555556,0.722222,0.811111
1,Arno Lamp Oil Merchants,261,2120284.44,0.131106,0.013975,6.968494,7,0.888889,0.444444,0.722222,0.722222
2,San Lorenzo Couriers,243,1956222.61,0.120961,0.018429,8.209836,7,0.555556,0.888889,0.722222,0.688889
3,Santa Maria Scribes,241,1963349.16,0.121402,0.022311,15.412372,6,0.666667,1.000000,0.277778,0.688889
4,Guildhall Security Company,257,1975651.98,0.122163,0.012914,6.127721,7,0.777778,0.111111,0.722222,0.566667
5,Florentine Paperworks,247,1831857.29,0.113271,0.013851,8.875419,7,0.444444,0.333333,0.722222,0.466667
6,Signoria Utilities Office,208,1353255.88,0.083677,0.017824,6.489388,6,0.222222,0.777778,0.277778,0.400000
7,Ponte Vecchio Rent Office,249,1753266.14,0.108412,0.013158,5.031168,7,0.333333,0.222222,0.722222,0.377778
8,Florence Operations,252,1018852.87,0.063000,0.017486,11.820767,4,0.111111,0.666667,0.111111,0.277778


Suspicious supplier candidate: Mercato Maintenance Works


## 7) Estimate Total Fraud Amount

Fraud estimate heuristic:
- For the suspicious supplier, compute monthly spend per debit account.
- Treat 5% vendor share as a normal upper baseline.
- Excess over that baseline is estimated suspicious amount.

In [10]:
sus = florence_expenses[florence_expenses['vendor'] == suspicious_supplier].copy()

monthly_account_total = (
    florence_expenses
    .groupby(['month', 'debit_account'], as_index=False)
    .agg(group_total=('debit_amount', 'sum'))
)

sus_monthly = (
    sus
    .groupby(['month', 'debit_account'], as_index=False)
    .agg(vendor_amount=('debit_amount', 'sum'), tx_count=('id', 'count'))
)

sus_monthly = sus_monthly.merge(monthly_account_total, on=['month', 'debit_account'], how='left')
sus_monthly['expected_max'] = 0.05 * sus_monthly['group_total']
sus_monthly['estimated_excess'] = (sus_monthly['vendor_amount'] - sus_monthly['expected_max']).clip(lower=0)

estimated_fraud_total = sus_monthly['estimated_excess'].sum()
display(sus_monthly.sort_values('estimated_excess', ascending=False).head(20))
print('Estimated total suspicious amount:', f'{estimated_fraud_total:,.2f}')

,month,debit_account,vendor_amount,tx_count,group_total,expected_max,estimated_excess
229,1438-11,Accounts Payable,87269.37,1,87269.37,4363.4685,82905.9015
109,1410-10,Accounts Payable,81614.60,2,81614.60,4080.7300,77533.8700
212,1434-01,Accounts Payable,70525.46,1,76181.19,3809.0595,66716.4005
66,1402-06,Accounts Payable,68807.74,2,68807.74,3440.3870,65367.3530
102,1410-01,Accounts Payable,54342.26,1,143826.22,7191.3110,47150.9490
121,1413-06,Accounts Payable,50021.51,1,59911.28,2995.5640,47025.9460
184,1425-04,Accounts Payable,50315.77,1,81892.45,4094.6225,46221.1475
114,1412-02,Accounts Payable,47291.70,1,47291.70,2364.5850,44927.1150
87,1407-02,Accounts Payable,45856.95,1,53131.54,2656.5770,43200.3730
56,1400-09,Accounts Payable,52538.38,2,195009.30,9750.4650,42787.9150


Estimated total suspicious amount: 2,022,455.41


## 8) Identify Fraud Date Range

In [11]:
suspicious_windows = sus_monthly[sus_monthly['estimated_excess'] > 0][['month', 'debit_account']]

if suspicious_windows.empty:
    print('No suspicious windows found under current assumptions.')
else:
    suspicious_tx = sus.merge(suspicious_windows, on=['month', 'debit_account'], how='inner')
    fraud_start = suspicious_tx['date'].min()
    fraud_end = suspicious_tx['date'].max()

    print('Suspicious supplier:', suspicious_supplier)
    print('Estimated fraud amount:', f'{estimated_fraud_total:,.2f}')
    print('Estimated fraud date range:', fraud_start.date(), 'to', fraud_end.date())
    print('Suspicious transaction count:', len(suspicious_tx))
    display(
        suspicious_tx[['id', 'date', 'debit_account', 'vendor', 'debit_amount', 'description']]
        .sort_values('date')
        .head(30)
    )

Suspicious supplier: Mercato Maintenance Works
Estimated fraud amount: 2,022,455.41
Estimated fraud date range: 1390-06-17 to 1440-10-14
Suspicious transaction count: 242


,id,date,debit_account,vendor,debit_amount,description
0,772,1390-06-17,Wages,Mercato Maintenance Works,433.91,Wages expense for Florence branch (monthly cycle)
1,920,1390-07-22,Maintenance,Mercato Maintenance Works,1661.61,Maintenance expense for Florence branch (quart...
2,1414,1390-11-04,Accounts Payable,Mercato Maintenance Works,6194.17,Vendor payment to Mercato Maintenance Works fo...
3,2069,1391-03-18,Security,Mercato Maintenance Works,289.85,Security expense for Florence branch (monthly ...
4,2623,1391-07-07,Accounts Payable,Mercato Maintenance Works,9386.31,Vendor payment to Mercato Maintenance Works fo...
5,3496,1392-01-17,Supplies,Mercato Maintenance Works,4019.83,Supplies expense for Florence branch (monthly ...
6,3536,1392-01-23,Rent,Mercato Maintenance Works,2337.56,Rent expense for Florence branch (monthly cycle)
7,3594,1392-02-04,Accounts Payable,Mercato Maintenance Works,1042.74,Vendor payment to Mercato Maintenance Works fo...
8,3751,1392-03-08,Accounts Payable,Mercato Maintenance Works,1604.13,Vendor payment to Mercato Maintenance Works fo...
9,3821,1392-03-23,Maintenance,Mercato Maintenance Works,2795.51,Maintenance expense for Florence branch (quart...


## Student Reflection Prompts
1. Which assumptions in this notebook might overstate or understate fraud?
2. How would your findings change if the concentration baseline were 10% instead of 5%?
3. Which additional controls or data fields would strengthen the investigation?